In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "BPQ_E.xpt"
csv_file = DATA_DIR / "BPQ_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Preserve untouched XPT for fidelity comparison
source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 BPQ_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BPQ020",
    "BPQ030",
    "BPD035",
    "BPQ040A",
    "BPQ050A",
    "BPQ052",
    "BPQ057",
    "BPQ060",
    "BPQ070",
    "BPQ080",
    "BPQ090A",
    "BPQ090B",
    "BPQ090C",
    "BPQ090D",
    "BPQ100A",
    "BPQ100B",
    "BPQ100C",
    "BPQ100D"
]


if df.shape != (6546, 19):

    raise ValueError(
        f"Unexpected BPQ_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "BPQ_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():

    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: BPQ_E structure = "
    "6,546 rows x 19 columns."
)


# ============================================================
# 4. CHECK ITEMS MUST NOT BE ADDED
# ============================================================

#
# CDC HTML contains:
#
# BPQ055 - CHECK ITEM
# BPQ095 - CHECK ITEM
#
# These are routing instructions, NOT released XPT variables.
#

CHECK_ITEMS = [
    "BPQ055",
    "BPQ095"
]


for col in CHECK_ITEMS:

    if col in df.columns:

        raise ValueError(
            f"Unexpected check-item column found: {col}"
        )


print(
    "PASS: BPQ055 and BPQ095 correctly absent "
    "from released XPT."
)


# ============================================================
# 5. KNOWN SAS/XPORT TINY VALUE CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPT TINY VALUE CHECK ---"
)

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifact found. "
        "Do NOT automatically replace it with zero."
    )


print(
    "PASS: BPQ_E contains NO tiny XPORT artifacts."
)


# ============================================================
# 6. TRUE NUMERIC ZERO CHECK
# ============================================================

true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\nTrue numeric zeros:",
    true_zero_total
)


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .to_string()
    )

    raise ValueError(
        "Unexpected source zero(s) found."
    )


print(
    "PASS: Original BPQ_E contains no numeric zeros."
)


# ============================================================
# 7. SEQN VALIDATION
# ============================================================

seqn = df["SEQN"]


if int(
    seqn.isna().sum()
) != 0:

    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated().sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41475:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "\nPASS: SEQN range = 41475-51623."
)

print(
    "PASS: Missing SEQN = 0."
)

print(
    "PASS: Duplicate SEQN = 0."
)


# ============================================================
# 8. FRACTIONAL / DECIMAL CHECK
# ============================================================

#
# BPQ_E should contain NO genuine decimal variables.
#

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:

        fractional_columns[
            col
        ] = fractional_count


print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)

print(
    "Genuine fractional variables:",
    fractional_columns
)


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal values found:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: Genuine decimal variables = 0."
)


# ============================================================
# 9. CDC CATEGORICAL FREQUENCY EXPECTATIONS
# ============================================================

#
# These are the exact values observed in the attached XPT
# and published in the CDC BPQ_E codebook.
#
# Missing values are checked separately.
#

EXPECTED_FREQUENCIES = {

    "BPQ020": {
        1: 2142,
        2: 4398,
        9: 6
    },

    "BPQ030": {
        1: 1780,
        2: 354,
        9: 8
    },

    "BPQ040A": {
        1: 1892,
        2: 249,
        9: 1
    },

    "BPQ050A": {
        1: 1671,
        2: 220,
        9: 1
    },

    "BPQ052": {
        1: 336,
        2: 6142,
        9: 68
    },

    "BPQ057": {
        1: 484,
        2: 5669,
        9: 56
    },

    "BPQ060": {
        1: 4218,
        2: 1518,
        7: 1,
        9: 198
    },

    "BPQ070": {
        1: 2715,
        2: 756,
        3: 491,
        4: 239,
        9: 17
    },

    "BPQ080": {
        1: 1906,
        2: 2277,
        9: 35
    },

    "BPQ090A": {
        1: 1540,
        2: 359,
        9: 7
    },

    "BPQ090B": {
        1: 1137,
        2: 765,
        9: 4
    },

    "BPQ090C": {
        1: 1300,
        2: 604,
        9: 2
    },

    "BPQ090D": {
        1: 1332,
        2: 569,
        9: 5
    },

    "BPQ100A": {
        1: 1298,
        2: 242
    },

    "BPQ100B": {
        1: 950,
        2: 187
    },

    "BPQ100C": {
        1: 965,
        2: 335
    },

    "BPQ100D": {
        1: 1085,
        2: 247
    }
}


EXPECTED_MISSING = {

    "BPQ020": 0,
    "BPQ030": 4404,
    "BPD035": 4422,
    "BPQ040A": 4404,
    "BPQ050A": 4654,
    "BPQ052": 0,
    "BPQ057": 337,
    "BPQ060": 611,
    "BPQ070": 2328,
    "BPQ080": 2328,
    "BPQ090A": 4640,
    "BPQ090B": 4640,
    "BPQ090C": 4640,
    "BPQ090D": 4640,
    "BPQ100A": 5006,
    "BPQ100B": 5409,
    "BPQ100C": 5246,
    "BPQ100D": 5214
}


# ============================================================
# 10. CHECK CATEGORICAL VARIABLES AGAINST CDC
# ============================================================

print(
    "\n--- CDC CATEGORICAL FREQUENCY CHECK ---"
)


for col, expected_counts in (
    EXPECTED_FREQUENCIES.items()
):

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v in actual_counts.items()
    }


    if actual_counts != expected_counts:

        raise ValueError(
            f"{col} frequency mismatch.\n"
            f"Expected: {expected_counts}\n"
            f"Actual:   {actual_counts}"
        )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    expected_missing = (
        EXPECTED_MISSING[col]
    )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{col} missing count mismatch.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 11. SPECIAL CDC CHECK - BPD035
# ============================================================

#
# Age first told of hypertension:
#
# CDC:
#
# 4-79 = 2,039
# 80   = 32
# 777  = 0
# 999  = 53
# Missing = 4,422
#

s = df["BPD035"]


age_range_count = int(
    s.between(
        4,
        79
    )
    .sum()
)


age_80_count = int(
    s.eq(80)
    .sum()
)


age_refused_count = int(
    s.eq(777)
    .sum()
)


age_dk_count = int(
    s.eq(999)
    .sum()
)


age_missing_count = int(
    s.isna()
    .sum()
)


if age_range_count != 2039:

    raise ValueError(
        f"BPD035 4-79 count differs: "
        f"{age_range_count:,}"
    )


if age_80_count != 32:

    raise ValueError(
        f"BPD035 age 80 count differs: "
        f"{age_80_count:,}"
    )


if age_refused_count != 0:

    raise ValueError(
        "Unexpected BPD035=777."
    )


if age_dk_count != 53:

    raise ValueError(
        "BPD035=999 count differs."
    )


if age_missing_count != 4422:

    raise ValueError(
        "BPD035 missing count differs."
    )


print(
    "\nPASS: BPD035 matches CDC."
)

print(
    "      4-79 = 2,039"
)

print(
    "      80   = 32"
)

print(
    "      999  = 53"
)

print(
    "      Missing = 4,422"
)


# ============================================================
# 12. EXPECTED DATASET FOR FIDELITY
# ============================================================

#
# BPQ_E requires NO numeric correction.
#
# Tiny artifacts = 0
# Zero restorations = 0
# Genuine decimals = 0
#
# Therefore expected_df is EXACTLY
# the original released XPT.
#

expected_df = source_df.copy()


# ============================================================
# 13. CLEAN WHOLE-NUMBER FORMATTING
# ============================================================

#
# All 19 released variables have been verified
# to contain whole-number identifiers/codes/ages.
#
# Examples:
#
# SEQN:
# 41475.0 -> 41475
#
# BPQ020:
# 1.0 -> 1
#
# BPD035:
# 50.0 -> 50
#
# BPQ060:
# 7.0 -> 7
#

for col in EXPECTED_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All BPQ_E variables converted "
    "to clean whole-number formatting."
)


# ============================================================
# 14. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    equal = np.isclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not bool(
        np.all(equal)
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export values equal "
    "original XPT exactly."
)


# ============================================================
# 15. EXPORT CSV
# ============================================================

#
# No:
#
# - rounding
# - float_format
# - recoding
# - filling missing values
#
# Missing XPT values -> blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 16. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 17. STRUCTURE CHECK AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 18. EXACT NUMERIC XPT -> CSV FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "XPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 19 columns numerically "
    "match original XPT exactly."
)


# ============================================================
# 19. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += (
        differences
    )


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing-position fidelity failure:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 20. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    raw_rows = list(
        csv.DictReader(f)
    )


if len(raw_rows) != 6546:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 21. STRICT RAW CSV FORMAT CHECK
# ============================================================

#
# Every nonmissing BPQ_E value should be:
#
# str(int(source_value))
#
# Examples:
#
# 41475.0 -> "41475"
# 1.0     -> "1"
# 80.0    -> "80"
# 999.0   -> "999"
#
# Missing -> ""
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:

        print(example)


    raise ValueError(
        "BPQ_E raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV tokens match expected "
    "whole-number representation exactly."
)


# ============================================================
# 22. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Unwanted '.0' values:",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        dot_zero_examples
    )

    raise ValueError(
        "Unwanted .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 23. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 24. TINY ARTIFACT STRING CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 25. FINAL CDC FREQUENCY RECHECK
# ============================================================

print(
    "\n--- FINAL FREQUENCY RECHECK ---"
)


for col, expected_counts in (
    EXPECTED_FREQUENCIES.items()
):

    actual_counts = (
        csv_df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v in actual_counts.items()
    }


    if actual_counts != expected_counts:

        raise ValueError(
            f"Final {col} frequencies changed."
        )


    if int(
        csv_df[col]
        .isna()
        .sum()
    ) != EXPECTED_MISSING[col]:

        raise ValueError(
            f"Final {col} missing count changed."
        )


print(
    "PASS: Final categorical frequencies "
    "still match original XPT/CDC."
)


# BPD035 final check

s = csv_df["BPD035"]


if not (
    int(
        s.between(
            4,
            79
        )
        .sum()
    ) == 2039

    and

    int(
        s.eq(80)
        .sum()
    ) == 32

    and

    int(
        s.eq(777)
        .sum()
    ) == 0

    and

    int(
        s.eq(999)
        .sum()
    ) == 53

    and

    int(
        s.isna()
        .sum()
    ) == 4422
):

    raise ValueError(
        "Final BPD035 distribution changed."
    )


print(
    "PASS: Final BPD035 distribution unchanged."
)


# ============================================================
# 26. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("BPQ_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nTiny artifacts:",
    tiny_total
)

print(
    "Tiny -> zero corrections:",
    0
)

print(
    "True source zeros:",
    true_zero_total
)

print(
    "Genuine decimal variables:",
    0
)


print(
    "\nCDC CHECK ITEMS:"
)

print(
    "BPQ055 = documentation/routing only"
)

print(
    "BPQ095 = documentation/routing only"
)

print(
    "Neither added to CSV"
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Unwanted .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "BPQ_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 BPQ_E XPT -> CSV
Rows: 6,546
Columns: 19
PASS: BPQ_E structure = 6,546 rows x 19 columns.
PASS: BPQ055 and BPQ095 correctly absent from released XPT.

--- ORIGINAL XPT TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: BPQ_E contains NO tiny XPORT artifacts.

True numeric zeros: 0
PASS: Original BPQ_E contains no numeric zeros.

PASS: SEQN range = 41475-51623.
PASS: Missing SEQN = 0.
PASS: Duplicate SEQN = 0.

--- FRACTIONAL VALUE CHECK ---
Genuine fractional variables: {}
PASS: Genuine decimal variables = 0.

--- CDC CATEGORICAL FREQUENCY CHECK ---
PASS: BPQ020
PASS: BPQ030
PASS: BPQ040A
PASS: BPQ050A
PASS: BPQ052
PASS: BPQ057
PASS: BPQ060
PASS: BPQ070
PASS: BPQ080
PASS: BPQ090A
PASS: BPQ090B
PASS: BPQ090C
PASS: BPQ090D
PASS: BPQ100A
PASS: BPQ100B
PASS: BPQ100C
PASS: BPQ100D

PASS: BPD035 matches CDC.
      4-79 = 2,039
      80   = 32
      999  = 53
      Missing = 4,422

PASS: All BPQ_E variables converted to clean whole-number formatting.
PASS: Pre-export val